This notebook contains the Sage scripts used to compute the probability bounds from the paper:

*Analyzing the Real-Work Security of the Algorand Blockchain*

to be published at ACM CCS 2023.

Eprint: https://eprint.iacr.org/2023/1344



**WARNING**: This is a [Sage](https://www.sagemath.org/) Jupyter notebook. This is not a pure Python Jupyter notebook.

Open using 

```bash
sage -n jupyter proba.ipynb
```

In [1]:
from scipy.stats import poisson, binom, hypergeom, norm
import numpy as np
import collections

## Parameters

This section just defines the various parameters we use.

In [2]:
Step = collections.namedtuple("Step", ["name", "E", "t"])
# E = size of committee
# t = threshold

steps = [
    Step("soft", 2990, 2267),
    Step("cert", 1500, 1112),
    Step("next", 5000, 3838),
    Step("late", 500, 320),
    Step("redo", 2400, 1768),
    Step("down", 6000, 4560),
    Step("prop", 20, 1)
]

# Matrix of configurations to test: each has a size multiplier, threshold ratio, and alpha
configurations = [
    {"mult": 1, "t": 0.76, "alpha": 0.20},
    # Increase alpha to 0.25
    {"mult": 1, "t": 0.76, "alpha": 0.25},
    # Decrease T/E
    {"mult": 1, "t": 0.72, "alpha": 0.25},
    # Multiply committee sizes
    {"mult": 2, "t": 0.72, "alpha": 0.25},
    {"mult": 3, "t": 0.72, "alpha": 0.25},
    # alpha = .30
    {"mult": 3, "t": 0.68, "alpha": 0.30},
    {"mult": 16, "t": 0.68, "alpha": 0.30},
    {"mult": 32, "t": 0.68, "alpha": 0.30},
    # High advesary stake
    {"mult": 1, "t": 0.76, "alpha": 0.37},
    {"mult": 3, "t": 0.72, "alpha": 0.37},
    {"mult": 32, "t": 0.68, "alpha": 0.37},
]

base_steps = steps  # Store original steps

def get_steps_for_config(config):
    size_mult = config["mult"]
    threshold_ratio = config["t"]

    name = ""

    if config["mult"] == 1 and config["t"] == 0.76:
        name = "Today's committee sizes and threshold (0.76)"
    else:
        name = f"{config['mult']}x committees with {config['t']:.2f} threshold"

    name += f" and {config['alpha']:.2f} adversary stake"
    print(f"\n{'='*60}")
    print(name)
    print(f"{'='*60}")
    new_steps = []
    for s in base_steps:
        new_E = int(s.E * size_mult)
        if s.name == "prop":
            new_t = 1
        else:
            new_t = int(new_E * threshold_ratio)
        new_steps.append(Step(s.name, new_E, new_t))
    return new_steps

## Safety/Validity Failures (Theorem 6.1 and Lemma A.20) - Events A.1-A.6

### Event A.1

This uses Lemma A.19.

In [3]:
Nmin = 1e12


def poi(x, k):
    return x**k / (factorial(k) * exp(x))

def step2failure_formal(E2, T2, alpha):
    """    
    Return the probability of failure and its log in base 2
    for the step 2 / soft vote step (expected size E2, quorum/threshold T2)
    That is the probability the adversary control X parties on the committee
    and there are Y honest parties on the committee so that:
    2 X + Y >= 2 * T2
    """
    
    p0 = E2/Nmin
    lamY = alpha*E2 + p0
    lamZ = E2*(1-alpha) + p0
    NN = 3*lamY # a bound for our sum, we use this value
    # we chose this value because we remark 
    # that the missing terms of the computation
    # are smaller than sum(poi(E2*alpha + E2/Nmin, i)) for i=NN,...
    # which is poisson(E2*alpha + E2/Nmin).sf(NN-1)
    # which is negligible:
    negl = poisson(lamY).sf(NN-1)
    assert(log(float(negl),2).n() < -256)
    # we anyway add those terms later to be on the safe side

    def too_many_honest_prob(num_mal):
        return poi(lamY, num_mal) * poisson(lamZ).sf(2*T2-2*num_mal-1)

    pf = sum(too_many_honest_prob(i) for i in (0..NN)) + negl
    pflog2 = log(float(pf), 2).n()
    
    return pf, pflog2

for config in configurations:
    alpha = config["alpha"]
    p = 1 - alpha
    steps = get_steps_for_config(config)
    named_steps = {step.name: step for step in steps}
    
    print(f"Probability of safety failure for soft-vote committee:")
    
    step = steps[0]
    name = step.name
    assert name == "soft"
    E2 = step.E
    T2 = step.t
    pf, pflog2 = step2failure_formal(E2, T2, alpha)
    print(f"  {name} E={E2:4d} t={T2:4d}: 2^{pflog2:.1f}")


Today's committee sizes and threshold (0.76) and 0.20 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=2990 t=2272: 2^-130.7

Today's committee sizes and threshold (0.76) and 0.25 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=2990 t=2272: 2^-87.9

1x committees with 0.72 threshold and 0.25 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=2990 t=2152: 2^-46.2

2x committees with 0.72 threshold and 0.25 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=5980 t=4305: 2^-88.9

3x committees with 0.72 threshold and 0.25 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=8970 t=6458: 2^-131.4

3x committees with 0.68 threshold and 0.30 adversary stake
Probability of safety failure for soft-vote committee:
  soft E=8970 t=6099: 2^-15.4

16x committees with 0.68 threshold and 0.30 adversary stake
Probability of safety failure for soft-vote c

### Events A.2 - A.6

This uses Lemma A.20. (We directly use Chernoff here as the bounds are already good enough this way.)

In [4]:
for config in configurations:
    alpha = config["alpha"]
    p = 1 - alpha
    steps = get_steps_for_config(config)
    named_steps = {step.name: step for step in steps}
    
    for step in steps[1:-1]:
        # using the new Chernoff bound from the paper,
        # e^{-(\alpha E-Q)^2/(\alpha E+Q)}  --- Q = T
        pf = exp(-(alpha * step.E - step.t)**2 / (alpha * step.E + step.t))
        pflog2 = log(pf, 2)
        pf250log2 = log(250 * pf, 2)
        pf250s = f" (250 iters union bound: 2^{pf250log2:.3f})" if step.name == "next" else ""
        print(f"  {step.name} E={step.E:4d} t={step.t:4d}: {pf:.5f}=2^{pflog2:.3f} {pf250s}")


Today's committee sizes and threshold (0.76) and 0.20 adversary stake
  cert E=1500 t=1140: 0.00000=2^-706.921 
  next E=5000 t=3800: 0.00000=2^-2356.402  (250 iters union bound: 2^-2348.436)
  late E= 500 t= 380: 0.00000=2^-235.640 
  redo E=2400 t=1824: 0.00000=2^-1131.073 
  down E=6000 t=4560: 0.00000=2^-2827.682 

Today's committee sizes and threshold (0.76) and 0.25 adversary stake
  cert E=1500 t=1140: 0.00000=2^-557.295 
  next E=5000 t=3800: 0.00000=2^-1857.648  (250 iters union bound: 2^-1849.683)
  late E= 500 t= 380: 0.00000=2^-185.765 
  redo E=2400 t=1824: 0.00000=2^-891.671 
  down E=6000 t=4560: 0.00000=2^-2229.178 

1x committees with 0.72 threshold and 0.25 adversary stake
  cert E=1500 t=1080: 0.00000=2^-492.822 
  next E=5000 t=3600: 0.00000=2^-1642.739  (250 iters union bound: 2^-1634.773)
  late E= 500 t= 360: 0.00000=2^-164.274 
  redo E=2400 t=1728: 0.00000=2^-788.515 
  down E=6000 t=4320: 0.00000=2^-1971.287 

2x committees with 0.72 threshold and 0.25 advers

## Liveness Failures (Theorem 6.2, Lemma A.23, and Corollary A.24) - Events B.1-B.5

This uses Lemma A.22.

In [5]:
def live_fail_bound(step, p):
    lambda_ = step.E * p
    pf = poisson(lambda_).cdf(step.t-1) * exp(((step.t-1)*lambda_*2+step.t-1)/ (2*minN*p))
    pflog2 = log(float(pf), 2).n()
    return pf, pflog2

minN = 1e12

for config in configurations:
    alpha = config["alpha"]
    p = 1 - alpha
    steps = get_steps_for_config(config)
    named_steps = {step.name: step for step in steps}
    
    print(f"Proven bounds of probability that not enough honest parties in committees (assuming N>{minN}):")
    for step in steps:
        pf, pflog2 = live_fail_bound(step, p)
        print(f"  {step.name} E={step.E:4d} t={step.t:4d}: {pf:.5f}=2^{pflog2:.1f}")
        
    print("B1.")
    c = 60
    pf_PV, _ = live_fail_bound(named_steps["prop"], p)
    pf_SV, _ = live_fail_bound(named_steps["soft"], p)
    pf_CV, _ = live_fail_bound(named_steps["cert"], p)
    pf_gc = (1 - (1 - alpha) * (1 - pf_PV)) + pf_SV + pf_CV
    pflog2_gc = log(float(pf_gc), 2).n() * c
    print(f"  2^{pflog2_gc}")
    
    print("B2.")
    c_ = 15
    pf_NV, _ = live_fail_bound(named_steps["next"], p)
    pf_nv = c * (pf_NV**(c_ + 1) + (c_ + 1) * pf_NV**(c_) * (1 - pf_NV) + (c_ * (c_ + 1)/2) * pf_NV**(c_-1) * (1 - pf_NV)**2)
    pflog2_nv = log(float(pf_nv), 2).n()
    print(f"  2^{pflog2_nv}")


Today's committee sizes and threshold (0.76) and 0.20 adversary stake
Proven bounds of probability that not enough honest parties in committees (assuming N>1.00000000000000e12):
  soft E=2990 t=2272: 0.00654=2^-7.3
  cert E=1500 t=1140: 0.03949=2^-4.7
  next E=5000 t=3800: 0.00070=2^-10.5
  late E= 500 t= 380: 0.15255=2^-2.7
  redo E=2400 t=1824: 0.01330=2^-6.2
  down E=6000 t=4560: 0.00023=2^-12.1
  prop E=  20 t=   1: 0.00000=2^-23.1
B1.
  2^-121.384689969070
B2.
  2^-133.907666226483

Today's committee sizes and threshold (0.76) and 0.25 adversary stake
Proven bounds of probability that not enough honest parties in committees (assuming N>1.00000000000000e12):
  soft E=2990 t=2272: 0.73059=2^-0.5
  cert E=1500 t=1140: 0.66871=2^-0.6
  next E=5000 t=3800: 0.79084=2^-0.3
  late E= 500 t= 380: 0.59503=2^-0.7
  redo E=2400 t=1824: 0.71111=2^-0.5
  down E=6000 t=4560: 0.81262=2^-0.3
  prop E=  20 t=   1: 0.00000=2^-21.6
B1.
  2^43.3112644152457
B2.
  2^4.25841475266925

1x committees wit